## > run

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import time

from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.preprocessing import MinMaxScaler

In [2]:
range_minpts = 100
local        = "100k2d"
ti           = 100
tf           = 1001
plot         = False
skip         = []

In [3]:
sns.set_context('poster')
sns.set_style('white')
sns.set_color_codes()

plot_kwds = {'s' : 10, 'linewidths':0}

In [4]:
minpts_min = 2
minpts_max = 201

In [5]:
df_time = {}

for minpts in range(minpts_min, minpts_max, 2):
    df_time[minpts] = pd.DataFrame(columns=["#_objects", "hdbscan_clusters", "corestream_clusters", "hastream_clusters", "hdbscan_outliers", "corestream_outliers", "hastream_outliers", "corestream_ari", "hastream_ari", "time"])

In [6]:
for t in range(ti, tf, ti):
    try:
        df_partition_corestream  = pd.read_csv("corestream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_bubbles.csv", sep=',')
        df_partition_hdbscan_cor = pd.read_csv("corestream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_hdbscan.csv", sep=',')
        
        df_partition_hastream    = pd.read_csv("hastream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_mcs.csv", sep=',')
        df_partition_hdbscan_has = pd.read_csv("hastream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_hdbscan.csv", sep=',')
    
        ARI_cor = pd.read_csv("corestream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/ARI_partitions.csv", sep=',', index_col=0)
        ARI_has = pd.read_csv("hastream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/ARI_partitions.csv", sep=',', index_col=0)
        
        data = pd.read_csv("corestream/results/" + local + "/datasets/data_t" + str(t) + ".csv", sep=',')
        
        print("t: ", t)
        
        for minpts in range(minpts_min, minpts_max, 2):
            
            # Make Directory
            m_directory = os.path.join(os.getcwd(), "results_study_ari/" + local + "/minpts_" + str(minpts))

            if not os.path.exists(m_directory):
                os.makedirs(m_directory)

            if plot:
                # ---------------------- PLOT ---------------------------
                fig, axs = plt.subplots(1, 3, figsize=(45, 15))

                fig.suptitle('Clusters from CoreStream, HASTREAM and HDBSCAN | dataset: ' + str(local) + " | MinPts: " + str(minpts) + " | time: " + str(t), fontsize=30)

                # CoreStream
                title  = ""
                title += "Corestream: " + " | " + "# Objects: " + str(len(df_partition_corestream.loc[(minpts / 2) - 1])) + " | "
                title += "# Clusters: " + str(df_partition_corestream.loc[(minpts / 2) - 1].max()) + " | "
                title += "# Outliers: " + str((df_partition_corestream.loc[(minpts / 2) - 1] == -1).sum()) + " | "
                title += "ARI: " + str(round(ARI_cor.at[minpts,'ARI'], 4))

                axs[0].set_title(title)
                axs[0].scatter(data['0'], data['1'], c=df_partition_corestream.loc[(minpts / 2) - 1], cmap='magma', **plot_kwds)

                # HDBSCAN_CORESTREAM
                title  = ""
                title += "HDBSCAN: " + " | " + "# Objects: " + str(len(df_partition_hdbscan_cor.loc[(minpts / 2) - 1])) + " | "
                title += "# Clusters: " + str(df_partition_hdbscan_cor.loc[(minpts / 2) - 1].max() + 1) + " | "
                title += "# Outliers: " + str((df_partition_hdbscan_cor.loc[(minpts / 2) - 1] == -1).sum())

                axs[1].set_title(title)
                axs[1].scatter(data['0'], data['1'], c=df_partition_hdbscan_cor.loc[(minpts / 2) - 1], cmap='magma', **plot_kwds)

                # HAStream
                title  = ""
                title += "HAStream: " + " | " + "# Objects: " + str(len(df_partition_hastream.loc[(minpts / 2) - 1])) + " | "
                title += "# Clusters: " + str(df_partition_hastream.loc[(minpts / 2) - 1].max()) + " | "
                title += "# Outliers: " + str((df_partition_hastream.loc[(minpts / 2) - 1] == -1).sum()) + " | "
                title += "ARI: " + str(round(ARI_has.at[minpts,'ARI'], 4))

                axs[2].set_title(title)
                axs[2].scatter(data['0'], data['1'], c=df_partition_hastream.loc[(minpts / 2) - 1], cmap='magma', **plot_kwds)

                plt.tight_layout()

                fig.savefig("results_study_ari/" + local + "/minpts_" + str(minpts) + "/minpts_" + str(minpts) + "_t_" + str(t) + ".png")

                plt.close()
                # ---------------------- PLOT ---------------------------
            
            infos = {
                "#_objects": len(df_partition_corestream.loc[(minpts / 2) - 1]),
                "hdbscan_clusters": df_partition_hdbscan_cor.loc[(minpts / 2) - 1].max() + 1,
                "corestream_clusters": df_partition_corestream.loc[(minpts / 2) - 1].max(),
                "hastream_clusters": df_partition_hastream.loc[(minpts / 2) - 1].max(),
                "hdbscan_outliers": (df_partition_hdbscan_cor.loc[(minpts / 2) - 1] == -1).sum(),
                "corestream_outliers": (df_partition_corestream.loc[(minpts / 2) - 1] == -1).sum(),
                "hastream_outliers": (df_partition_hastream.loc[(minpts / 2) - 1] == -1).sum(),
                "corestream_ari": round(ARI_cor.at[minpts,'ARI'], 4),
                "hastream_ari": round(ARI_has.at[minpts,'ARI'], 4),
                "time": t
            }
            
            df_time[minpts].loc[t] = infos
            
    except FileNotFoundError:
        print(f"Erro: O arquivo do tempo (" + str(t) + ") não foi encontrado.") 

t:  100
t:  200
t:  300
t:  400
t:  500
t:  600
t:  700
t:  800
t:  900
t:  1000


In [7]:
for minpts in range(minpts_min, minpts_max, 2):
    df_time[minpts].to_csv("results_study_ari/" + local + "/minpts_" + str(minpts) + "/infos.csv", index=False)

### Plot line clusters

In [8]:
# Make Directory
m_directory = os.path.join(os.getcwd(), "results_study_ari/" + local + "/all_plot_infos")

if not os.path.exists(m_directory):
    os.makedirs(m_directory)

for minpts in range(minpts_min, minpts_max, 2):
    print("MinPts: ", minpts)
    
    #df = pd.read_csv("results_study_ari/" + local + "/minpts_" + str(minpts) + "/infos.csv", sep=',')
    df = df_time[minpts]
    
    fig, axs = plt.subplots(3, 1, figsize=(35, 25))

    fig.suptitle('Clusters from CoreStream, HASTREAM and HDBSCAN | dataset: ' + str(local) + " | MinPts: " + str(minpts), fontsize=30)

    # ---------------- CORESTREAM -----------------------
    diference = (df['hdbscan_clusters'] - df['corestream_clusters']).abs()
    axs[0].bar(df['time'].mean() - ti, diference.mean(), yerr=diference.std(), label='Mean and Str Dif(COR-HDB) Clusters', capsize=5, align='center', alpha=0.7, ecolor='black', color="green", width=40)
    diference = (df['hdbscan_clusters'] - df['hastream_clusters']).abs()
    axs[0].bar(df['time'].mean() + ti, diference.mean(), yerr=diference.std(), label='Mean and Str Dif(HAS-HDB) Clusters', capsize=5, align='center', alpha=0.7, ecolor='black', color="red", width=40)
    axs[0].plot(df['time'], df['hdbscan_clusters'], 'bs--', label='HDBSCAN Clusters')
    axs[0].plot(df['time'], df['corestream_clusters'], 'go--', label='COREStream Clusters')
    axs[0].plot(df['time'], df['hastream_clusters'], 'r^--', label='HAStream Clusters')
    axs[0].set_title("COREStream and HDBSCAN")
    axs[0].set_ylabel("Clusters", fontsize=25)
    axs[0].set_xlabel("time", fontsize=30)
    axs[0].legend(loc='center left', bbox_to_anchor=(1, 0.5), shadow=True)
    
    # ---------------- OUTLIERS -------------------------
    diference = (df['hdbscan_outliers'] - df['corestream_outliers']).abs()
    axs[1].bar(df['time'].mean() - ti, diference.mean(), yerr=diference.std(), label='Mean and Str Dif(COR-HDB) Outliers', capsize=5, align='center', alpha=0.7, ecolor='black', color="green", width=40)
    diference = (df['hdbscan_outliers'] - df['hastream_outliers']).abs()
    axs[1].bar(df['time'].mean() + ti, diference.mean(), yerr=diference.std(), label='Mean and Str Dif(HAS-HDB) Outliers', capsize=5, align='center', alpha=0.7, ecolor='black', color="red", width=40)
    axs[1].plot(df['time'], df['hdbscan_outliers'], 'bs--', label='HDBSCAN Outliers')
    axs[1].plot(df['time'], df['hastream_outliers'], 'r^--', label='HASTREAM Outliers')
    axs[1].plot(df['time'], df['corestream_outliers'], 'go--', label='CORESTREAM Outliers')
    axs[1].set_title("Outliers COREStream, HAStream and HDBSCAN")
    axs[1].set_ylabel("Outliers", fontsize=25)
    axs[1].set_xlabel("time", fontsize=30)
    axs[1].legend(loc='center left', bbox_to_anchor=(1, 0.5), shadow=True)

    # ---------------- HASTREAM -------------------------
    #diference = (df['hdbscan_clusters'] - df['hastream_clusters']).abs()
    #axs[1].plot(df['time'], df['hdbscan_clusters'], 'bs--')
    #axs[1].plot(df['time'], df['hastream_clusters'], 'r^--')
    #axs[1].bar(df['time'].mean(), diference.mean(), yerr=diference.std(), label='Mean and Str clusters', capsize=5, align='center', alpha=0.7, ecolor='black', color="red", width=40)
    #axs[1].set_title("HAStream and HDBSCAN")
    #axs[1].set_ylabel("Clusters", fontsize=25)
    #axs[1].set_xlabel("time", fontsize=30)
    #axs[1].legend(('HDBSCAN', 'HAStream'), loc=0, shadow=True)

    # ---------------- ARI -------------------------
    axs[2].plot(df['time'], df['corestream_ari'], 'go--', label='CORESTREAM ARI')
    axs[2].plot(df['time'], df['hastream_ari'], 'r^--', label='HASTREAM ARI')
    axs[2].bar(df['time'].mean() - ti, df['corestream_ari'].mean(), yerr=df['corestream_ari'].std(), label='CORESTREAM Mean and Str ARI', capsize=5, align='center', alpha=0.7, ecolor='black', color="green", width=40)
    axs[2].bar(df['time'].mean() + ti, df['hastream_ari'].mean(), yerr=df['hastream_ari'].std(), label='HASTREAM Mean and Str ARI', capsize=5, align='center', alpha=0.7, ecolor='black', color="red", width=40)
    axs[2].set_title("ARI per time, COREStream and HAStream")
    axs[2].set_ylabel("ARI", fontsize=25)
    axs[2].set_xlabel("time", fontsize=30)
    axs[2].legend(loc='center left', bbox_to_anchor=(1, 0.5), shadow=True)

    # Add vertical dashed lines at each x value
    for x_value in df['time']:
        axs[0].axvline(x=x_value, color='gray', linestyle='--', linewidth=1)
        axs[1].axvline(x=x_value, color='gray', linestyle='--', linewidth=1)
        axs[2].axvline(x=x_value, color='gray', linestyle='--', linewidth=1)
        #axs[3].axvline(x=x_value, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()

    fig.savefig("results_study_ari/" + local + "/all_plot_infos/infos_minpts_" + str(minpts) + ".png")

    plt.close()

MinPts:  2
MinPts:  4
MinPts:  6
MinPts:  8
MinPts:  10
MinPts:  12
MinPts:  14
MinPts:  16
MinPts:  18
MinPts:  20
MinPts:  22
MinPts:  24
MinPts:  26
MinPts:  28
MinPts:  30
MinPts:  32
MinPts:  34
MinPts:  36
MinPts:  38
MinPts:  40
MinPts:  42
MinPts:  44
MinPts:  46
MinPts:  48
MinPts:  50
MinPts:  52
MinPts:  54
MinPts:  56
MinPts:  58
MinPts:  60
MinPts:  62
MinPts:  64
MinPts:  66
MinPts:  68
MinPts:  70
MinPts:  72
MinPts:  74
MinPts:  76
MinPts:  78
MinPts:  80
MinPts:  82
MinPts:  84
MinPts:  86
MinPts:  88
MinPts:  90
MinPts:  92
MinPts:  94
MinPts:  96
MinPts:  98
MinPts:  100
MinPts:  102
MinPts:  104
MinPts:  106
MinPts:  108
MinPts:  110
MinPts:  112
MinPts:  114
MinPts:  116
MinPts:  118
MinPts:  120
MinPts:  122
MinPts:  124
MinPts:  126
MinPts:  128
MinPts:  130
MinPts:  132
MinPts:  134
MinPts:  136
MinPts:  138
MinPts:  140
MinPts:  142
MinPts:  144
MinPts:  146
MinPts:  148
MinPts:  150
MinPts:  152
MinPts:  154
MinPts:  156
MinPts:  158
MinPts:  160
MinPts:  162


plt.figure(figsize=(10, 8))

plt.bar('HDBSCAN', df['hdbscan_clusters'].mean(), yerr=df['hdbscan_clusters'].std(), capsize=5, align='center', alpha=0.7, ecolor='black', color="blue", label='HDBSCAN')
plt.bar('COREStream', df['corestream_clusters'].mean(), yerr=df['corestream_clusters'].std(), capsize=5, align='center', alpha=0.7, ecolor='black', color="orange", label='COREStream')
#plt.bar('HAStream', df['hastream_clusters'].mean(), yerr=df['hastream_clusters'].std(), capsize=5, align='center', alpha=0.7, ecolor='black', color="green", label='HAStream')

plt.ylabel("Clusters", fontsize=14)
plt.xlabel("time", fontsize=14)

plt.legend(loc=0, shadow=True)

plt.show()

### plot CoreStream, HDBSCAN and HAStream partitions

start_time = time.time()

for t in range(ti, tf, ti):
    try:
        df_partition_corestream  = pd.read_csv("corestream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_bubbles.csv", sep=',')
        df_partition_hdbscan_cor = pd.read_csv("corestream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_hdbscan.csv", sep=',')
        
        df_partition_hastream    = pd.read_csv("hastream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_mcs.csv", sep=',')
        df_partition_hdbscan_has = pd.read_csv("hastream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/all_partitions_hdbscan.csv", sep=',')
        
        ARI_cor = pd.read_csv("corestream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/ARI_partitions.csv", sep=',', index_col=0)
        ARI_has = pd.read_csv("hastream/results/" + local + "/flat_solutions/flat_solution_partitions_t" + str(t) + "/ARI_partitions.csv", sep=',', index_col=0)
        
        data = pd.read_csv("corestream/results/" + local + "/datasets/data_t" + str(t) + ".csv", sep=',')
        
        print("t: ", t)
        
        for minpts in range(0, range_minpts):
            fig, axs = plt.subplots(1, 3, figsize=(45, 15))

            fig.suptitle('Clusters from CoreStream, HASTREAM and HDBSCAN | dataset: ' + str(local) + " | MinPts: " + str((minpts + 1) * 2), fontsize=30)

            #CoreStream
            title  = ""
            title += "Corestream: " + " | " + "# Objects: " + str(len(df_partition_corestream.loc[minpts])) + " | "
            title += "# Clusters: " + str(df_partition_corestream.loc[minpts].max()) + " | "
            title += "# Outliers: " + str((df_partition_corestream.loc[minpts] == -1).sum()) + " | "
            title += "ARI: " + str(round(ARI_cor.at[(minpts + 1) * 2,'ARI'], 4))

            axs[0].set_title(title)
            axs[0].scatter(data['0'], data['1'], c=df_partition_corestream.loc[minpts], cmap='magma', **plot_kwds)

            # HDBSCAN_CORESTREAM
            title  = ""
            title += "HDBSCAN: " + " | " + "# Objects: " + str(len(df_partition_hdbscan_cor.loc[minpts])) + " | "
            title += "# Clusters: " + str(df_partition_hdbscan_cor.loc[minpts].max() + 1) + " | "
            title += "# Outliers: " + str((df_partition_hdbscan_cor.loc[minpts] == -1).sum())

            axs[1].set_title(title)
            axs[1].scatter(data['0'], data['1'], c=df_partition_hdbscan_cor.loc[minpts], cmap='magma', **plot_kwds)
            
            #HAStream
            title  = ""
            title += "HAStream: " + " | " + "# Objects: " + str(len(df_partition_hastream.loc[minpts])) + " | "
            title += "# Clusters: " + str(df_partition_hastream.loc[minpts].max()) + " | "
            title += "# Outliers: " + str((df_partition_hastream.loc[minpts] == -1).sum()) + " | "
            title += "ARI: " + str(round(ARI_has.at[(minpts + 1) * 2,'ARI'], 4))

            axs[2].set_title(title)
            axs[2].scatter(data['0'], data['1'], c=df_partition_hastream.loc[minpts], cmap='magma', **plot_kwds)

            plt.tight_layout()

            m_directory = os.path.join(os.getcwd(), "results_study_ari/" + local + "/time_t" + str(t))

            if not os.path.exists(m_directory):
                os.makedirs(m_directory)

            fig.savefig("results_study_ari/" + local + "/time_t" + str(t) + "/mints_" + str((minpts + 1) * 2) + ".png")

            plt.close()
        
    except FileNotFoundError:
        print(f"Erro: O arquivo não foi encontrado.")

print("Total Time: ", time.time() - start_time)